# Dataset Integration: Bone Tumor Detection

This notebook prepares a unified metadata table for Dataset 1 and BTXRD. The target task is binary **Normal** versus **Tumor** classification, while each dataset's original class is preserved.

The workflow does not resize, normalize, augment, delete, modify, merge, or synthesize images. It also does not apply SMOTE or train a model. Missing or ambiguous information is reported rather than inferred.

## 1. Imports and paths

Both datasets should be placed somewhere below `data/raw/`. Set `DATASET_1_ROOT` and `BTXRD_ROOT` if their locations are known. Otherwise, the notebook searches directory names and reports what it finds.

In [ ]:
from collections import defaultdict
from hashlib import sha256
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
FIGURES_ROOT = PROJECT_ROOT / 'results' / 'figures'
for folder in (PROCESSED_ROOT, FIGURES_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
DATASET_1_ROOT = Path(os.environ['DATASET_1_ROOT']).expanduser() if os.environ.get('DATASET_1_ROOT') else None
BTXRD_ROOT = Path(os.environ['BTXRD_ROOT']).expanduser() if os.environ.get('BTXRD_ROOT') else None
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data root exists: {RAW_ROOT.exists()} ({RAW_ROOT})')

## 2. Discover the actual directory and file structure

This is an inspection step. No folder layout is assumed. CSV files and image files are listed recursively, and likely dataset roots are identified from directory names where possible.

In [ ]:
def list_structure(root, limit=200):
    if not root or not root.exists():
        return []
    return [path.relative_to(root).as_posix() for path in sorted(root.rglob('*')) if path.is_file()][:limit]

all_files = list_structure(RAW_ROOT)
print(f'Files found under data/raw/: {len(all_files)}')
for relative_path in all_files:
    print(relative_path)

candidate_dirs = [path for path in RAW_ROOT.rglob('*') if path.is_dir()] if RAW_ROOT.exists() else []
if DATASET_1_ROOT is None:
    matches = [path for path in candidate_dirs if any(token in path.name.lower() for token in ('dataset1', 'dataset_1', 'cancer', 'bone'))]
    DATASET_1_ROOT = matches[0] if matches else None
if BTXRD_ROOT is None:
    matches = [path for path in candidate_dirs if 'btxrd' in path.name.lower()]
    BTXRD_ROOT = matches[0] if matches else None

print(f'\nDataset 1 root: {DATASET_1_ROOT}')
print(f'BTXRD root: {BTXRD_ROOT}')
if DATASET_1_ROOT is None or BTXRD_ROOT is None:
    print('One or both dataset roots could not be determined. Set DATASET_1_ROOT and BTXRD_ROOT before running integration.')

## 3. Read labels and verify referenced images

CSV schemas can differ, so the notebook detects likely image and label columns. If a CSV cannot be interpreted, it is recorded as an issue. If no usable CSV is found, class-folder names are used only as a fallback and are marked as folder-derived labels.

In [ ]:
IMAGE_COLUMN_NAMES = ('image_path', 'image', 'filename', 'file_name', 'file', 'path', 'filepath', 'image_id')
LABEL_COLUMN_NAMES = ('label', 'class', 'category', 'diagnosis', 'target', 'y')

def image_files(root):
    return sorted(path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS) if root and root.exists() else []

def choose_column(columns, candidates):
    normalized = {str(column).strip().lower(): column for column in columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for column in columns:
        lowered = str(column).lower()
        if any(candidate in lowered for candidate in candidates):
            return column
    return None

def resolve_image(reference, root, images):
    reference = str(reference).strip()
    candidates = [Path(reference), root / reference, root / Path(reference).name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = [path for path in images if path.name == Path(reference).name]
    return matches[0].resolve() if len(matches) == 1 else None

def records_from_dataset(root, source_name):
    issues = []
    images = image_files(root)
    records = []
    csv_files = sorted(root.rglob('*.csv')) if root and root.exists() else []
    for csv_path in csv_files:
        try:
            table = pd.read_csv(csv_path)
        except Exception as error:
            issues.append(f'{source_name}: could not read {csv_path}: {error}')
            continue
        image_column = choose_column(table.columns, IMAGE_COLUMN_NAMES)
        label_column = choose_column(table.columns, LABEL_COLUMN_NAMES)
        if image_column is None or label_column is None:
            issues.append(f'{source_name}: skipped {csv_path.name}; image or label column was not identified')
            continue
        for _, row in table[[image_column, label_column]].dropna().iterrows():
            image_path = resolve_image(row[image_column], root, images)
            original_label = str(row[label_column]).strip()
            records.append({'image_path': str(image_path) if image_path else str(row[image_column]), 'source_dataset': source_name, 'original_label': original_label, 'label_source': f'CSV:{csv_path.name}', 'image_exists': image_path is not None})
    if not records:
        for image_path in images:
            relative_parts = image_path.relative_to(root).parts
            original_label = next((part for part in relative_parts[:-1] if part.lower() in {'normal', 'cancer', 'benign', 'malignant', 'tumor'}), 'UNKNOWN')
            records.append({'image_path': str(image_path.resolve()), 'source_dataset': source_name, 'original_label': original_label, 'label_source': 'folder name fallback', 'image_exists': True})
        if images and not csv_files:
            issues.append(f'{source_name}: no CSV files found; labels were derived from folder names')
    return records, issues

dataset_records = []
findings = []
if DATASET_1_ROOT and DATASET_1_ROOT.exists():
    records, issues = records_from_dataset(DATASET_1_ROOT, 'Dataset 1')
    dataset_records.extend(records); findings.extend(issues)
else:
    findings.append('Dataset 1 root is missing or could not be determined.')
if BTXRD_ROOT and BTXRD_ROOT.exists():
    records, issues = records_from_dataset(BTXRD_ROOT, 'BTXRD')
    dataset_records.extend(records); findings.extend(issues)
else:
    findings.append('BTXRD root is missing or could not be determined.')

metadata = pd.DataFrame(dataset_records, columns=['image_path', 'source_dataset', 'original_label', 'label_source', 'image_exists'])
print(f'Referenced image records: {len(metadata):,}')
display(metadata.head())

## 4. Apply the proposed binary label mapping

Dataset 1 maps `Cancer` to `Tumor`. BTXRD maps both `Benign` and `Malignant` to `Tumor`. The original label remains unchanged in `original_label`; an unknown label is never silently assigned to either target class.

In [ ]:
def harmonize_label(source_dataset, original_label):
    label = str(original_label).strip().lower()
    if label == 'normal':
        return 'Normal'
    if source_dataset == 'Dataset 1' and label == 'cancer':
        return 'Tumor'
    if source_dataset == 'BTXRD' and label in {'benign', 'malignant'}:
        return 'Tumor'
    return pd.NA

if not metadata.empty:
    metadata['binary_label'] = [harmonize_label(source, label) for source, label in zip(metadata['source_dataset'], metadata['original_label'])]
    unknown_labels = metadata[metadata['binary_label'].isna()]
else:
    metadata['binary_label'] = pd.Series(dtype='string')
    unknown_labels = metadata
print('Unknown or unmapped labels:', len(unknown_labels))
display(metadata['binary_label'].value_counts(dropna=False).rename_axis('binary_label').to_frame('image_count'))

## 5. Image readability and hash-based duplicate checks

Images are opened only for validation and hashing. Their pixels and files are not changed. Exact duplicates are marked in metadata, including whether the duplicate occurs within one dataset or across datasets.

In [ ]:
def inspect_image(path):
    try:
        with Image.open(path) as image:
            image.load()
            return {'readable': True, 'width': image.width, 'height': image.height, 'format': image.format, 'color_mode': image.mode, 'error': ''}
    except (OSError, UnidentifiedImageError, ValueError) as error:
        return {'readable': False, 'width': pd.NA, 'height': pd.NA, 'format': '', 'color_mode': '', 'error': str(error)}

def file_hash(path):
    digest = sha256()
    with open(path, 'rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

image_checks = []
for path_text in metadata['image_path'] if not metadata.empty else []:
    path = Path(path_text)
    check = inspect_image(path) if path.is_file() else {'readable': False, 'width': pd.NA, 'height': pd.NA, 'format': '', 'color_mode': '', 'error': 'File does not exist'}
    check['sha256'] = file_hash(path) if check['readable'] else pd.NA
    image_checks.append(check)

if not metadata.empty:
    metadata = pd.concat([metadata.reset_index(drop=True), pd.DataFrame(image_checks)], axis=1)
    hash_counts = metadata.dropna(subset=['sha256']).groupby('sha256').size()
    metadata['duplicate_group'] = metadata['sha256'].map(hash_counts).fillna(0).astype(int)
    metadata['is_duplicate'] = metadata['duplicate_group'] > 1
    dataset_counts = metadata.groupby('sha256')['source_dataset'].nunique()
    cross_dataset_hashes = set(dataset_counts[dataset_counts > 1].index)
    metadata['duplicate_scope'] = np.where(metadata['sha256'].isin(cross_dataset_hashes), 'across datasets', np.where(metadata['is_duplicate'], 'within dataset', 'none'))
else:
    for column in ('readable', 'width', 'height', 'format', 'color_mode', 'error', 'sha256', 'duplicate_group', 'is_duplicate', 'duplicate_scope'):
        metadata[column] = pd.Series(dtype='object')

missing_or_unreadable = metadata[~metadata['readable']] if not metadata.empty else metadata
duplicate_rows = metadata[metadata['is_duplicate']] if not metadata.empty else metadata
cross_dataset_duplicates = duplicate_rows[duplicate_rows['duplicate_scope'] == 'across datasets'] if not duplicate_rows.empty else duplicate_rows
print(f'Missing or unreadable images: {len(missing_or_unreadable)}')
print(f'Duplicate image records: {len(duplicate_rows)}')
print(f'Cross-dataset duplicate records: {len(cross_dataset_duplicates)}')
display(metadata.head())

## 6. Original and harmonized distributions

These tables show the evidence without deleting duplicate, missing, corrupt, or unmapped records. Counts are based on the records discovered from the supplied files.

In [ ]:
original_distribution = (metadata.groupby(['source_dataset', 'original_label'], dropna=False).size().reset_index(name='image_count')) if not metadata.empty else pd.DataFrame(columns=['source_dataset', 'original_label', 'image_count'])
summary_table = (metadata.groupby(['source_dataset', 'original_label', 'binary_label'], dropna=False).size().reset_index(name='image_count')) if not metadata.empty else pd.DataFrame(columns=['source_dataset', 'original_label', 'binary_label', 'image_count'])
binary_distribution = metadata['binary_label'].value_counts(dropna=False).rename_axis('binary_label').reset_index(name='image_count') if not metadata.empty else pd.DataFrame(columns=['binary_label', 'image_count'])
display(original_distribution)
display(summary_table)
display(binary_distribution)

def add_percentages(table, group_columns):
    result = table.copy()
    if result.empty:
        result['percentage'] = pd.Series(dtype=float)
        return result
    totals = result.groupby(group_columns)['image_count'].transform('sum')
    result['percentage'] = (result['image_count'] / totals * 100).round(2)
    return result

print('Original class percentages by source:')
display(add_percentages(original_distribution, ['source_dataset']))
print('Unified class percentages:')
display(add_percentages(binary_distribution, []))

## 7. Plots and metadata export

Plots are saved under `results/figures/`. The metadata CSV includes all discovered records and duplicate/readability flags; no rows are silently removed.

In [ ]:
sns.set_theme(style='whitegrid')
if not original_distribution.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=original_distribution, x='original_label', y='image_count', hue='source_dataset')
    plt.title('Original class distribution by dataset')
    plt.xlabel('Original class')
    plt.ylabel('Image count')
    plt.tight_layout()
    plt.savefig(FIGURES_ROOT / 'original_class_distribution.png', dpi=150)
    plt.show()

if not binary_distribution.empty:
    plt.figure(figsize=(6, 4))
    sns.barplot(data=binary_distribution.dropna(subset=['binary_label']), x='binary_label', y='image_count', color='#4C78A8')
    plt.title('Unified binary class distribution')
    plt.xlabel('Binary label')
    plt.ylabel('Image count')
    plt.tight_layout()
    plt.savefig(FIGURES_ROOT / 'unified_binary_distribution.png', dpi=150)
    plt.show()

metadata_path = PROCESSED_ROOT / 'unified_metadata.csv'
metadata.to_csv(metadata_path, index=False)
print(f'Saved metadata to: {metadata_path}')
print('Saved figures to:', FIGURES_ROOT)

## 8. Integration Decision

The datasets should be combined only after reviewing the evidence below. In particular, confirm that the CSV-to-file references are complete, images are readable, class semantics are documented, duplicate handling is agreed, and the proposed Benign/Malignant-to-Tumor mapping is appropriate for the research question. This notebook does not make that decision automatically.

Duplicates are retained and marked in metadata. A later data-preparation step should define a documented policy, such as excluding one copy from a training manifest while preserving the original files and audit trail. Unreadable, missing, or unmapped records require review before model development.

In [ ]:
print('INTEGRATION SUMMARY')
print(f'Total integrated image records: {len(metadata):,}')
normal_count = int((metadata['binary_label'] == 'Normal').sum())
tumor_count = int((metadata['binary_label'] == 'Tumor').sum())
print(f'Normal count: {normal_count:,}')
print(f'Tumor count: {tumor_count:,}')
print('Counts by source dataset:')
print(metadata['source_dataset'].value_counts(dropna=False).to_string() if not metadata.empty else 'No records found.')
print('Counts by original class:')
print(metadata['original_label'].value_counts(dropna=False).to_string() if not metadata.empty else 'No records found.')
print(f'Duplicate findings: {len(duplicate_rows):,} duplicate records; {len(cross_dataset_duplicates):,} cross-dataset duplicate records.')
print(f'Missing/corrupt image findings: {len(missing_or_unreadable):,}')
print('Other findings:')
for finding in findings:
    print('-', finding)
if unknown_labels.shape[0] > 0:
    print(f'- {len(unknown_labels)} records have labels that were not mapped and require review.')
print('Decision status: evidence has been prepared; no automatic merge decision was made.')